<a href="https://colab.research.google.com/github/FJWangYantao/Pytorch-/blob/main/%E8%87%AA%E5%8A%A8%E5%BE%AE%E5%88%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 图表设置
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\| **Autograd** \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

用 `torch.autograd` 进行自动微分
===============================================

训练神经网络时，**后向传播**是最常用的算法。在这个算法中参数（模型权重）通过损失函数在特定参数的**梯度**调整。

为计算梯度，PyTorch 提供了一个内置的 `torch.autograd`自动微分引擎。为任意计算图提供自动梯度计算。

考察最简单的单层神经网络，输入 `x` `w` `b` 和一些损失函数，可以在 Pytorch 中以下面方式定义：


In [3]:
import torch

x = torch.ones(5)  # 输入层
y = torch.zeros(3)  # 预期输出
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

张量，函数，计算图
==========================================

以上代码定义了如下**计算图**：

![](https://pytorch.org/tutorials/_static/img/basics/comp-graph.png)

在这个网络中 `w` 和 `b` 都是需要被优化的**参数**。因此我们需要参照这些变量来计算梯度。Pytorch 设置了`requires_grad`这个属性。


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>创建张量时可以设置 <code>requires_grad</code> 这个参数, 或者创建后通过调用 <code>x.requires_grad_(True)</code> 这个方法</p>

</div>



用于构建计算图的函数实际上是一个`Function`的对象。这个对象能自动解析前向传播的方向和如何计算后向传播过程的导数。后向传播函数的引用被存储在张量的`grad_fn`属性中。可以在[这个文档](https://pytorch.org/docs/stable/autograd.html#function)中获取更多关于`Function`的信息。


In [4]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x785e3f0b1d20>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x785e3f0b0730>


计算梯度
===================

为优化神经网络中的权重，需要根据损失函数计算对应参数的导数，也就是说我们需要求出$\frac{\partial loss}{\partial w}$ 和 $\frac{\partial loss}{\partial b}$
为了计算这些导数我们调用 `loss.backward()` 再通过 `w.grad` 和d `b.grad` 取值


In [5]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.0100, 0.2554, 0.1616],
        [0.0100, 0.2554, 0.1616],
        [0.0100, 0.2554, 0.1616],
        [0.0100, 0.2554, 0.1616],
        [0.0100, 0.2554, 0.1616]])
tensor([0.0100, 0.2554, 0.1616])


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<ul>
<li>通过计算图的叶节点取得 <code>grad</code> 。 所求梯度的参数的<code>requires_grad</code> 参数被设为 <code>True</code>. 而其他图节点的参数不可获得。我们可以通过在给定的图上后向传播来进行梯度计算。如果我们需要用同一个计算图来进行多次后向传播，应该设置<code>retain_graph=True</code>在backward参数上。</li>
</ul>
```

</div>



禁用梯度跟踪
===========================

默认情况下所有采用`requires_grad=True`的张量是可以追踪计算历史和支持梯度计算的。但在有些情况下，比如我们已经训练好模型了，想要测试它在一组输入上的表现，我们此时只想要模型的前向传播计算。所以通过`torch.no_grad()`来在特定一块代码禁用梯度跟踪


In [ ]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

Another way to achieve the same result is to use the `detach()` method
on the tensor:


In [ ]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

There are reasons you might want to disable gradient tracking:

:   -   To mark some parameters in your neural network as **frozen
        parameters**.
    -   To **speed up computations** when you are only doing forward
        pass, because computations on tensors that do not track
        gradients would be more efficient.


More on Computational Graphs
============================

Conceptually, autograd keeps a record of data (tensors) and all executed
operations (along with the resulting new tensors) in a directed acyclic
graph (DAG) consisting of
[Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function)
objects. In this DAG, leaves are the input tensors, roots are the output
tensors. By tracing this graph from roots to leaves, you can
automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

-   run the requested operation to compute a resulting tensor
-   maintain the operation's *gradient function* in the DAG.

The backward pass kicks off when `.backward()` is called on the DAG
root. `autograd` then:

-   computes the gradients from each `.grad_fn`,
-   accumulates them in the respective tensor's `.grad` attribute
-   using the chain rule, propagates all the way to the leaf tensors.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>An important thing to note is that the graph is recreated from scratch; after each<code>.backward()</code> call, autograd starts populating a new graph. This isexactly what allows you to use control flow statements in your model;you can change the shape, size and operations at every iteration ifneeded.</p>

</div>



Optional Reading: Tensor Gradients and Jacobian Products
========================================================

In many cases, we have a scalar loss function, and we need to compute
the gradient with respect to some parameters. However, there are cases
when the output function is an arbitrary tensor. In this case, PyTorch
allows you to compute so-called **Jacobian product**, and not the actual
gradient.

For a vector function $\vec{y}=f(\vec{x})$, where
$\vec{x}=\langle x_1,\dots,x_n\rangle$ and
$\vec{y}=\langle y_1,\dots,y_m\rangle$, a gradient of $\vec{y}$ with
respect to $\vec{x}$ is given by **Jacobian matrix**:

$$\begin{aligned}
J=\left(\begin{array}{ccc}
   \frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{1}}{\partial x_{n}}\\
   \vdots & \ddots & \vdots\\
   \frac{\partial y_{m}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
   \end{array}\right)
\end{aligned}$$

Instead of computing the Jacobian matrix itself, PyTorch allows you to
compute **Jacobian Product** $v^T\cdot J$ for a given input vector
$v=(v_1 \dots v_m)$. This is achieved by calling `backward` with $v$ as
an argument. The size of $v$ should be the same as the size of the
original tensor, with respect to which we want to compute the product:


In [ ]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

Notice that when we call `backward` for the second time with the same
argument, the value of the gradient is different. This happens because
when doing `backward` propagation, PyTorch **accumulates the
gradients**, i.e. the value of computed gradients is added to the `grad`
property of all leaf nodes of computational graph. If you want to
compute the proper gradients, you need to zero out the `grad` property
before. In real-life training an *optimizer* helps us to do this.


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Previously we were calling <code>backward()</code> function withoutparameters. This is essentially equivalent to calling<code>backward(torch.tensor(1.0))</code>, which is a useful way to compute thegradients in case of a scalar-valued function, such as loss duringneural network training.</p>

</div>



------------------------------------------------------------------------


Further Reading
===============

-   [Autograd
    Mechanics](https://pytorch.org/docs/stable/notes/autograd.html)
